In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/abortion_1month.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What predicts a positive vs negative abortion experience? Is it the method, the support system, or the clinical environment?"

# What Predicts a Positive Abortion Experience?
## Method, Support System, or Clinical Environment

**Abstract:** Using 9,885 posts from 2,052 users on r/abortion (March 13 -- April 12, 2026), we analyzed the text of community posts to identify what factors predict whether someone describes their abortion experience positively. We examined three candidate predictors: abortion method (medical vs. surgical), social support (partner, family, friend), and clinical environment (staff quality, sedation). Of these three, only **staff quality** showed a statistically significant association with positive experience in pairwise testing: users who described kind, compassionate staff reported positive outcomes at 45.6% vs. 34.2% for those with no staff mention (Fisher's exact p=0.029, OR=1.61). Neither abortion method (p=1.00) nor social support (p=0.42) reached significance. A multivariate logistic regression confirmed this pattern, with positive staff as the only near-significant predictor (OR=1.52, p=0.06). The data suggest that most factors patients agonize over -- method choice, having a support person -- do not measurably predict experience quality in this community. What does matter, modestly, is how staff treat patients. Sample: 460 users with analyzable outcome signals from 2,052 total.

## Data Exploration

This analysis uses post text from r/abortion rather than treatment reports. The treatment_reports table captures medication sentiment (misoprostol, ibuprofen, etc.), but the research question is about the *experience itself* -- emotional trajectory, interpersonal support, and clinical environment. The richest signal for these themes lives in the body text of posts.

We use keyword-based theme extraction to tag each user's posts for the presence of method mentions, support indicators, staff quality descriptors, and emotional outcomes. All analysis is aggregated to the user level (one data point per user) to preserve statistical independence.

**Causal-context exclusions:** Birth control and Plan B are excluded from treatment analysis -- negative sentiment toward these reflects *why* users are in the community (contraceptive failure), not their abortion experience.

In [ ]:

import math
import re

total_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) FROM posts", conn).iloc[0,0]
total_posts = pd.read_sql("SELECT COUNT(*) FROM posts", conn).iloc[0,0]
date_range = pd.read_sql("SELECT MIN(date(post_date, 'unixepoch')) as mn, MAX(date(post_date, 'unixepoch')) as mx FROM posts", conn)
mn, mx = date_range.iloc[0]

flair_df = pd.read_sql(
    "SELECT flair, COUNT(DISTINCT user_id) as users FROM posts WHERE flair IS NOT NULL GROUP BY flair ORDER BY users DESC",
    conn
)

posts = pd.read_sql(
    "SELECT p.post_id, p.user_id, p.body_text, p.title, p.flair, p.post_date "
    "FROM posts p WHERE LENGTH(p.body_text) > 50 AND p.body_text NOT LIKE '%Welcome to /r/abortion%'",
    conn
)

bt = posts['body_text'].str.lower()
title_lower = posts['title'].fillna('').str.lower()

# Method tags
posts['mentions_surgical'] = (
    bt.str.contains('surgical|in-clinic|procedure room', regex=True, na=False)
    | title_lower.str.contains('surgical', regex=True, na=False)
)
posts['mentions_medical'] = (
    bt.str.contains('medication abortion|medical abortion|mifepristone|misoprostol|abortion pill|at home', regex=True, na=False)
    | title_lower.str.contains('medical abortion|medication|mifepristone|misoprostol', regex=True, na=False)
)

# Support tags
posts['mentions_partner'] = bt.str.contains(r'partner|boyfriend|husband|fiance', regex=True, na=False)
posts['mentions_family'] = bt.str.contains(r'\bmom\b|mother|\bdad\b|father|\bsister\b|\bbrother\b|family|parent', regex=True, na=False)
posts['mentions_friend'] = bt.str.contains(r'\bfriend\b|best friend|roommate', regex=True, na=False)
posts['mentions_alone'] = bt.str.contains(r'by myself|all alone|did it alone|no one knew|kept it secret|no support|completely alone', regex=True, na=False)
posts['has_support_person'] = posts['mentions_partner'] | posts['mentions_family'] | posts['mentions_friend']

# Clinical environment tags
posts['staff_positive'] = bt.str.contains(r'kind|gentle|compassionate|caring|nice nurse|wonderful staff|amazing staff|so nice|great staff|sweet nurse|lovely nurse|warm', regex=True, na=False)
posts['staff_negative'] = bt.str.contains(r'\brude\b|judgmental|\bcold\b|dismissive|uncaring|mean nurse|horrible staff|unprofessional', regex=True, na=False)
posts['mentions_pp'] = bt.str.contains(r'planned parenthood', regex=True, na=False)
posts['mentions_sedation'] = bt.str.contains(r'sedation|sedated|anesthesia|twilight', regex=True, na=False)

# Outcome tags
posts['expresses_relief'] = bt.str.contains(r'\brelief\b|\brelieved\b|weight off|burden lifted', regex=True, na=False)
posts['expresses_no_regret'] = bt.str.contains(r"no regret|right decision|best decision|zero regret|don't regret|do not regret|never regret|glad i did", regex=True, na=False)
posts['expresses_regret'] = (
    bt.str.contains(r'\bregret\b|wrong decision|\bmistake\b', regex=True, na=False)
    & ~bt.str.contains(r"no regret|not regret|never regret|zero regret|don't regret|do not regret|won't regret", regex=True, na=False)
)
posts['expresses_fear'] = bt.str.contains(r'\bscared\b|terrified|\bafraid\b|\bnervous\b|\banxious\b|\bpanic\b|\bfear\b', regex=True, na=False)
posts['pain_severe'] = bt.str.contains(r'worst pain|excruciating|unbearable|agony|screaming|so much pain|intense pain|horrible pain', regex=True, na=False)
posts['pain_mild'] = bt.str.contains(r'not as bad|manageable|like a period|mild cramp|not that bad|barely felt|less painful than', regex=True, na=False)

# Aggregate to user level
agg_cols = [
    'mentions_surgical', 'mentions_medical', 'mentions_partner', 'mentions_family',
    'mentions_friend', 'mentions_alone', 'has_support_person', 'staff_positive',
    'staff_negative', 'mentions_pp', 'mentions_sedation', 'expresses_relief',
    'expresses_no_regret', 'expresses_regret', 'expresses_fear', 'pain_severe', 'pain_mild',
]
user_themes = posts.groupby('user_id')[agg_cols].any().reset_index()

user_themes['positive_experience'] = (
    (user_themes['expresses_relief'] | user_themes['expresses_no_regret'])
    & ~user_themes['expresses_regret']
)
user_themes['negative_experience'] = user_themes['expresses_regret'] | user_themes['pain_severe']
user_themes['has_outcome_signal'] = (
    user_themes['expresses_relief'] | user_themes['expresses_no_regret']
    | user_themes['expresses_regret'] | user_themes['pain_severe'] | user_themes['pain_mild']
)

analyzable = user_themes[user_themes['has_outcome_signal']].copy()

display(HTML(
    "<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>"
    "<h3 style='margin-top:0;'>Dataset Overview</h3>"
    f"<p><strong>Data covers:</strong> {mn} to {mx} (1 month)</p>"
    f"<p><strong>Total users:</strong> {total_users:,} &nbsp;|&nbsp; <strong>Total posts:</strong> {total_posts:,}</p>"
    f"<p><strong>Users with analyzable experience signals:</strong> {len(analyzable):,} ({100*len(analyzable)/total_users:.0f}% of all users)</p>"
    f"<p><strong>Positive experience rate (relief/no-regret without regret):</strong> {analyzable['positive_experience'].mean():.1%}</p>"
    f"<p><strong>Geographic reach:</strong> {len(flair_df)} regions represented "
    f"(USA: {flair_df.iloc[0]['users']:,}, Asia: {flair_df.iloc[1]['users']:,}, UK/Ireland: {flair_df.iloc[2]['users']:,})</p>"
    "</div>"
))


## The Emotional Landscape of r/abortion

Before examining what predicts a positive experience, we need to understand the baseline: what emotions does this community express most often?

In [ ]:

emotion_counts = {
    'Fear/Anxiety': user_themes['expresses_fear'].sum(),
    'Mentions Support Person': user_themes['has_support_person'].sum(),
    'Mentions Partner': user_themes['mentions_partner'].sum(),
    'Mentions Family': user_themes['mentions_family'].sum(),
    'Regret (negated)': user_themes['expresses_no_regret'].sum(),
    'Regret (affirmed)': user_themes['expresses_regret'].sum(),
    'Relief': user_themes['expresses_relief'].sum(),
    'Staff Positive': user_themes['staff_positive'].sum(),
    'Severe Pain': user_themes['pain_severe'].sum(),
    'Mild Pain': user_themes['pain_mild'].sum(),
    'Staff Negative': user_themes['staff_negative'].sum(),
    'Alone': user_themes['mentions_alone'].sum(),
}
emotion_df = pd.DataFrame({
    'Theme': list(emotion_counts.keys()),
    'Users': list(emotion_counts.values()),
    'Pct': [100 * v / len(user_themes) for v in emotion_counts.values()]
}).sort_values('Users', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
pos_set = {'Relief', 'Regret (negated)', 'Staff Positive', 'Mild Pain', 'Mentions Support Person'}
neg_set = {'Fear/Anxiety', 'Regret (affirmed)', 'Staff Negative', 'Severe Pain', 'Alone'}
colors = ['#2ecc71' if t in pos_set else '#e74c3c' if t in neg_set else '#3498db'
          for t in emotion_df['Theme']]
bars = ax.barh(emotion_df['Theme'], emotion_df['Users'], color=colors)
for bar, pct in zip(bars, emotion_df['Pct']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Number of Users')
ax.set_title('Emotional and Contextual Themes in r/abortion Posts\n'
             '(Green = positive signal, Red = negative signal, Blue = neutral context)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

display(HTML(
    "<p><strong>Key takeaway:</strong> Fear and anxiety dominate the community -- nearly half of users "
    "with analyzable signals express it. But mentions of support people are nearly as common, and "
    "positive staff experiences outpace negative ones roughly 5:1. Among users who discuss regret, "
    'those <em>negating</em> it ("no regret," "right decision") outnumber those affirming it. '
    "This community skews toward processing difficult emotions rather than expressing "
    "lasting negative outcomes.</p>"
))


## Does Abortion Method Predict Experience?

The community discusses two primary methods: medical abortion (MA -- mifepristone/misoprostol, typically taken at home) and surgical abortion (SA -- an in-clinic procedure). Does choosing one method over the other predict whether a user describes their experience positively?

We identify method by keyword matching in post text. Users who mention both are tagged as "both." Users who mention neither are excluded from this comparison.

In [ ]:

method_users = analyzable[analyzable['mentions_surgical'] | analyzable['mentions_medical']].copy()
method_users['method'] = 'Both'
method_users.loc[method_users['mentions_medical'] & ~method_users['mentions_surgical'], 'method'] = 'Medical (MA)'
method_users.loc[method_users['mentions_surgical'] & ~method_users['mentions_medical'], 'method'] = 'Surgical (SA)'

method_summary = method_users.groupby('method').agg(
    n=('positive_experience', 'count'),
    positive=('positive_experience', 'sum'),
    negative=('negative_experience', 'sum'),
    pain_severe=('pain_severe', 'sum'),
    pain_mild=('pain_mild', 'sum'),
).reset_index()
method_summary['pos_rate'] = method_summary['positive'] / method_summary['n']
for idx, row in method_summary.iterrows():
    lo, hi = wilson_ci(int(row['positive']), int(row['n']))
    method_summary.loc[idx, 'ci_lo'] = lo
    method_summary.loc[idx, 'ci_hi'] = hi

ma_users = method_users[method_users['method'] == 'Medical (MA)']
sa_users = method_users[method_users['method'] == 'Surgical (SA)']

table_method = [
    [int(ma_users['positive_experience'].sum()), int((~ma_users['positive_experience']).sum())],
    [int(sa_users['positive_experience'].sum()), int((~sa_users['positive_experience']).sum())]
]
or_method, p_method = fisher_exact(table_method)
p1_ma = ma_users['positive_experience'].mean()
p2_sa = sa_users['positive_experience'].mean() if len(sa_users) > 0 else 0.001
cohens_h_method = 2 * (math.asin(math.sqrt(p1_ma)) - math.asin(math.sqrt(max(p2_sa, 0.001))))

# Forest plot
fig, ax = plt.subplots(figsize=(10, 4))
methods_plot = method_summary[method_summary['n'] >= 10].sort_values('pos_rate')
y_pos = range(len(methods_plot))
ax.errorbar(methods_plot['pos_rate'], y_pos,
            xerr=[methods_plot['pos_rate'] - methods_plot['ci_lo'],
                  methods_plot['ci_hi'] - methods_plot['pos_rate']],
            fmt='o', color='#2c3e50', capsize=5, markersize=8, linewidth=2)
ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{row['method']} (n={int(row['n'])})" for _, row in methods_plot.iterrows()])
ax.set_xlabel('Positive Experience Rate (with 95% Wilson CI)')
ax.set_title('Positive Experience Rate by Abortion Method')
baseline = analyzable['positive_experience'].mean()
ax.axvline(x=baseline, color='grey', linestyle='--', alpha=0.5, label='Overall baseline')
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

display(HTML(
    "<p><strong>Statistical comparison (Medical vs. Surgical):</strong><br>"
    f"Medical abortion positive rate: {p1_ma:.1%} (n={len(ma_users)})<br>"
    f"Surgical abortion positive rate: {p2_sa:.1%} (n={len(sa_users)})<br>"
    f"Fisher's exact OR = {or_method:.2f}, p = {p_method:.3f} | Cohen's h = {cohens_h_method:.3f}<br><br>"
    "<strong>Plain-language verdict:</strong> The rates are virtually identical -- method does not predict "
    "experience quality. The wide overlapping confidence intervals confirm there is no meaningful "
    "difference between medical and surgical abortion in terms of whether users report a positive experience.</p>"
))


## Does Social Support Predict Experience?

Method showed no predictive power. What about having someone to lean on? We compare users who mention a support person (partner, family member, friend) to those who do not mention one. Note that "not mentioning support" is not the same as "being alone" -- many users simply do not discuss their support situation.

In [ ]:

# Broader comparison: mentions support vs does not
sup_yes = analyzable[analyzable['has_support_person']]
sup_no = analyzable[~analyzable['has_support_person']]

table_support = [
    [int(sup_yes['positive_experience'].sum()), int((~sup_yes['positive_experience']).sum())],
    [int(sup_no['positive_experience'].sum()), int((~sup_no['positive_experience']).sum())]
]
or_support, p_support = fisher_exact(table_support)
p_sup = sup_yes['positive_experience'].mean()
p_nosup = sup_no['positive_experience'].mean()
h_support = 2 * (math.asin(math.sqrt(max(p_sup, 0.001))) - math.asin(math.sqrt(max(p_nosup, 0.001))))

# Support type breakdown
support_types = []
for _, row in analyzable.iterrows():
    if row['mentions_partner']:
        support_types.append({'user_id': row['user_id'], 'support_type': 'Partner', 'positive': row['positive_experience']})
    if row['mentions_family']:
        support_types.append({'user_id': row['user_id'], 'support_type': 'Family', 'positive': row['positive_experience']})
    if row['mentions_friend']:
        support_types.append({'user_id': row['user_id'], 'support_type': 'Friend', 'positive': row['positive_experience']})
support_type_df = pd.DataFrame(support_types) if support_types else pd.DataFrame(columns=['support_type', 'positive'])

sup_summary = support_type_df.groupby('support_type').agg(
    n=('positive', 'count'), pos=('positive', 'sum')
).reset_index()
sup_summary['rate'] = sup_summary['pos'] / sup_summary['n']
for idx, row in sup_summary.iterrows():
    lo, hi = wilson_ci(int(row['pos']), int(row['n']))
    sup_summary.loc[idx, 'ci_lo'] = lo
    sup_summary.loc[idx, 'ci_hi'] = hi

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.5]})

# Left: binary comparison
ax1 = axes[0]
cats = ['Mentions\nSupport', 'No Support\nMentioned']
rates = [p_sup, p_nosup]
ns = [len(sup_yes), len(sup_no)]
ci_sup = wilson_ci(int(sup_yes['positive_experience'].sum()), len(sup_yes))
ci_nosup = wilson_ci(int(sup_no['positive_experience'].sum()), len(sup_no))
bar_colors = ['#2ecc71', '#95a5a6']
yerr_lo = [rates[0] - ci_sup[0], rates[1] - ci_nosup[0]]
yerr_hi = [ci_sup[1] - rates[0], ci_nosup[1] - rates[1]]
bars1 = ax1.bar(cats, rates, color=bar_colors, width=0.6,
                yerr=[yerr_lo, yerr_hi], capsize=5, ecolor='#333')
for bar, n in zip(bars1, ns):
    ax1.text(bar.get_x() + bar.get_width()/2, min(bar.get_height() + 0.06, 0.94),
             f'n={n}', ha='center', fontsize=10)
ax1.set_ylabel('Positive Experience Rate')
ax1.set_title('Support Mentioned vs. Not')
ax1.set_ylim(0, 1.05)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: support type breakdown
ax2 = axes[1]
if len(sup_summary) > 0:
    sup_sorted = sup_summary.sort_values('rate', ascending=True)
    ax2.barh(
        [f"{row['support_type']} (n={int(row['n'])})" for _, row in sup_sorted.iterrows()],
        sup_sorted['rate'],
        xerr=[sup_sorted['rate'] - sup_sorted['ci_lo'], sup_sorted['ci_hi'] - sup_sorted['rate']],
        color='#3498db', capsize=4, ecolor='#333', height=0.6
    )
    ax2.set_xlabel('Positive Experience Rate')
    ax2.set_title('Positive Experience by Support Type')
    ax2.set_xlim(0, 1)
    ax2.axvline(x=analyzable['positive_experience'].mean(), color='grey', linestyle='--', alpha=0.5)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

display(HTML(
    "<p><strong>Statistical comparison (Mentions Support vs. Does Not):</strong><br>"
    f"Mentions support: {p_sup:.1%} positive (n={len(sup_yes)})<br>"
    f"No support mentioned: {p_nosup:.1%} positive (n={len(sup_no)})<br>"
    f"Fisher's exact OR = {or_support:.2f}, p = {p_support:.3f} | Cohen's h = {h_support:.3f}<br><br>"
    "<strong>Plain-language verdict:</strong> Users who mention a support person trend slightly higher in "
    "positive experience rates, but the difference is not statistically significant. "
    "The confidence intervals overlap substantially. We cannot conclude from this data that having "
    "a support person measurably improves experience quality -- though the direction is consistent "
    "with what we would expect.</p>"
))


## Does Clinical Environment Predict Experience?

Support and method both failed to reach significance. What about the clinical environment itself -- specifically, how staff behave?

This is where the strongest signal lives. We compare users who describe positive staff interactions to those who do not mention staff quality at all.

In [ ]:

# Staff positive vs no staff mention (cleanest comparison)
staff_pos_u = analyzable[analyzable['staff_positive']]
no_staff_u = analyzable[~analyzable['staff_positive'] & ~analyzable['staff_negative']]
staff_neg_u = analyzable[analyzable['staff_negative']]

table_staff = [
    [int(staff_pos_u['positive_experience'].sum()), int((~staff_pos_u['positive_experience']).sum())],
    [int(no_staff_u['positive_experience'].sum()), int((~no_staff_u['positive_experience']).sum())]
]
or_staff, p_staff = fisher_exact(table_staff)
p_pos_staff = staff_pos_u['positive_experience'].mean()
p_no_staff = no_staff_u['positive_experience'].mean()
p_neg_staff = staff_neg_u['positive_experience'].mean() if len(staff_neg_u) > 0 else 0
h_staff = 2 * (math.asin(math.sqrt(max(p_pos_staff, 0.001))) - math.asin(math.sqrt(max(p_no_staff, 0.001))))

# Also test staff_pos vs staff_neg directly
if len(staff_neg_u) >= 5:
    table_staff_direct = [
        [int(staff_pos_u['positive_experience'].sum()), int((~staff_pos_u['positive_experience']).sum())],
        [int(staff_neg_u['positive_experience'].sum()), int((~staff_neg_u['positive_experience']).sum())]
    ]
    or_direct, p_direct = fisher_exact(table_staff_direct)
else:
    or_direct, p_direct = float('nan'), float('nan')

# Heatmap: clinical factors x outcomes
clinical_factors = ['staff_positive', 'staff_negative', 'mentions_pp', 'mentions_sedation']
outcomes = ['positive_experience', 'expresses_relief', 'expresses_regret', 'pain_severe', 'pain_mild']
factor_labels_hm = ['Positive Staff', 'Negative Staff', 'Planned Parenthood', 'Sedation']
outcome_labels_hm = ['Positive\nExperience', 'Relief', 'Regret', 'Severe\nPain', 'Mild\nPain']

heatmap_data = np.zeros((len(clinical_factors), len(outcomes)))
heatmap_ns = np.zeros((len(clinical_factors),), dtype=int)
for i, f in enumerate(clinical_factors):
    filt = analyzable[analyzable[f]]
    heatmap_ns[i] = len(filt)
    for j, o in enumerate(outcomes):
        heatmap_data[i, j] = filt[o].mean() * 100 if len(filt) > 0 else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1.3, 1]})

# Heatmap
ax_heat = axes[0]
im = ax_heat.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=80)
ax_heat.set_xticks(range(len(outcome_labels_hm)))
ax_heat.set_xticklabels(outcome_labels_hm, fontsize=9)
ax_heat.set_yticks(range(len(factor_labels_hm)))
ax_heat.set_yticklabels([f"{fl} (n={heatmap_ns[i]})" for i, fl in enumerate(factor_labels_hm)], fontsize=10)
for i in range(len(clinical_factors)):
    for j in range(len(outcomes)):
        val = heatmap_data[i, j]
        ax_heat.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=9,
                     color='white' if val > 50 or val < 15 else 'black')
cbar = fig.colorbar(im, ax=ax_heat, shrink=0.8, pad=0.02)
cbar.set_label('Percentage of Users (%)', fontsize=9)
ax_heat.set_title('Clinical Factors vs. Outcome Themes')

# Right panel: grouped bar for staff comparison
ax2 = axes[1]
groups = ['Positive Staff', 'No Staff Mention', 'Negative Staff']
vals = [p_pos_staff, p_no_staff, p_neg_staff]
ns_staff = [len(staff_pos_u), len(no_staff_u), len(staff_neg_u)]
cis = [wilson_ci(int(staff_pos_u['positive_experience'].sum()), len(staff_pos_u)),
       wilson_ci(int(no_staff_u['positive_experience'].sum()), len(no_staff_u)),
       wilson_ci(int(staff_neg_u['positive_experience'].sum()), max(len(staff_neg_u), 1))]
colors_s = ['#2ecc71', '#95a5a6', '#e74c3c']
yerr_lo_s = [v - ci[0] for v, ci in zip(vals, cis)]
yerr_hi_s = [ci[1] - v for v, ci in zip(vals, cis)]
bars2 = ax2.bar(groups, vals, color=colors_s, width=0.6,
                yerr=[yerr_lo_s, yerr_hi_s], capsize=5, ecolor='#333')
for bar, n in zip(bars2, ns_staff):
    ax2.text(bar.get_x() + bar.get_width()/2, min(bar.get_height() + 0.05, 0.93),
             f'n={n}', ha='center', fontsize=10)
ax2.set_ylabel('Positive Experience Rate')
ax2.set_title('Staff Quality and\nPositive Experience')
ax2.set_ylim(0, 1.05)
ax2.axhline(y=analyzable['positive_experience'].mean(), color='grey', linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

display(HTML(
    "<p><strong>Statistical comparison (Positive Staff vs. No Staff Mention):</strong><br>"
    f"Positive staff: {p_pos_staff:.1%} positive (n={len(staff_pos_u)})<br>"
    f"No staff mention: {p_no_staff:.1%} positive (n={len(no_staff_u)})<br>"
    f"Fisher's exact OR = {or_staff:.2f}, p = {p_staff:.3f} | Cohen's h = {h_staff:.3f}<br><br>"
    f"<strong>Direct comparison (Positive vs. Negative staff):</strong> "
    f"Positive staff: {p_pos_staff:.1%}, Negative staff: {p_neg_staff:.1%} (n={len(staff_neg_u)}). "
    f"Fisher's exact p = {p_direct:.3f}. The negative staff group is too small (n={len(staff_neg_u)}) "
    f"for a reliable comparison.<br><br>"
    "<strong>Plain-language verdict:</strong> This is the only statistically significant finding in the "
    "analysis. Users who describe kind, compassionate staff are 11 percentage points more likely to "
    "report a positive experience than those who do not mention staff quality. The effect size is "
    f"small-to-moderate (Cohen's h = {h_staff:.2f}). In practical terms, for every ~9 patients who "
    "encounter positive staff, roughly 1 additional patient reports a positive experience compared "
    "to the baseline.</p>"
))


## Putting It All Together: Multivariate Model

The pairwise tests above suggest staff quality is the only significant predictor. A logistic regression tests whether this holds when all factors are included simultaneously, and reveals whether any factor is masking another.

In [ ]:

import statsmodels.api as sm

model_df = analyzable.copy()
model_df['method_surgical'] = model_df['mentions_surgical'].astype(int)
model_df['method_medical'] = model_df['mentions_medical'].astype(int)
model_df['has_support'] = model_df['has_support_person'].astype(int)
model_df['staff_pos'] = model_df['staff_positive'].astype(int)
model_df['staff_neg'] = model_df['staff_negative'].astype(int)
model_df['sedation'] = model_df['mentions_sedation'].astype(int)
model_df['fear'] = model_df['expresses_fear'].astype(int)
model_df['alone'] = model_df['mentions_alone'].astype(int)

features = ['method_surgical', 'method_medical', 'has_support', 'staff_pos',
            'staff_neg', 'sedation', 'fear', 'alone']
feature_labels = ['Surgical Method', 'Medical Method', 'Has Support Person',
                  'Positive Staff', 'Negative Staff', 'Sedation/Anesthesia',
                  'Expresses Fear', 'Alone/Secret']

X = model_df[features].values
y = model_df['positive_experience'].astype(int).values
X_const = sm.add_constant(X)

logit_model = sm.Logit(y, X_const).fit(disp=0)
coefs = logit_model.params[1:]
pvals = logit_model.pvalues[1:]
conf = logit_model.conf_int()[1:]
odds_ratios = np.exp(coefs)
or_ci_lo = np.exp(conf[:, 0])
or_ci_hi = np.exp(conf[:, 1])

results_df = pd.DataFrame({
    'Predictor': feature_labels,
    'Odds Ratio': odds_ratios,
    'CI Low': or_ci_lo,
    'CI High': or_ci_hi,
    'p-value': pvals,
    'Significant': ['*' if p < 0.05 else '' for p in pvals]
}).sort_values('Odds Ratio', ascending=False)

# Forest plot of odds ratios
fig, ax = plt.subplots(figsize=(10, 6))
res_sorted = results_df.sort_values('Odds Ratio')
y_pos = range(len(res_sorted))
colors_or = [
    '#2ecc71' if (p < 0.05 and o > 1) else '#e74c3c' if (p < 0.05 and o < 1)
    else '#f39c12' if (p < 0.10 and o > 1) else '#95a5a6'
    for o, p in zip(res_sorted['Odds Ratio'], res_sorted['p-value'])
]
ax.errorbar(res_sorted['Odds Ratio'], y_pos,
            xerr=[res_sorted['Odds Ratio'] - res_sorted['CI Low'],
                  res_sorted['CI High'] - res_sorted['Odds Ratio']],
            fmt='none', capsize=4, linewidth=1.5, color='#333')
for i, (_, row) in enumerate(res_sorted.iterrows()):
    ax.scatter(row['Odds Ratio'], i, color=colors_or[i], s=100, zorder=5, edgecolors='#333')
ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{row['Predictor']} (p={row['p-value']:.2f})"
                    for _, row in res_sorted.iterrows()])
ax.axvline(x=1.0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('Odds Ratio (95% CI)')
ax.set_title('Logistic Regression: Predictors of Positive Experience\n'
             '(Green = sig. p<0.05, Orange = marginal p<0.10, Grey = not significant)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_elts = [Patch(facecolor='#2ecc71', label='Significant (p<0.05)'),
               Patch(facecolor='#f39c12', label='Marginal (p<0.10)'),
               Patch(facecolor='#95a5a6', label='Not significant')]
ax.legend(handles=legend_elts, loc='lower right', frameon=True)
plt.tight_layout()
plt.show()

# Sensitivity: drop 3 most prolific users
user_post_counts = posts.groupby('user_id').size().reset_index(name='n_posts')
top3_users = user_post_counts.nlargest(3, 'n_posts')['user_id']
sens_df = model_df[~model_df['user_id'].isin(top3_users)]
X_s = sens_df[features].values
y_s = sens_df['positive_experience'].astype(int).values
X_s_const = sm.add_constant(X_s)
try:
    sens_model = sm.Logit(y_s, X_s_const).fit(disp=0)
    sens_pvals = dict(zip(feature_labels, sens_model.pvalues[1:]))
    main_pvals = dict(zip(feature_labels, pvals))
    # Check if staff_pos stays marginal
    staff_still_marginal = sens_pvals.get('Positive Staff', 1) < 0.10
    sens_note = ("Positive Staff remains the strongest predictor after dropping the 3 most prolific "
                 f"users (p={sens_pvals.get('Positive Staff', 1):.3f}). Findings are robust."
                 if staff_still_marginal else
                 f"After dropping 3 most prolific users, Positive Staff p-value changed to "
                 f"{sens_pvals.get('Positive Staff', 1):.3f} -- interpret with caution.")
except Exception:
    sens_note = "Sensitivity check could not be completed."

display(HTML(f"<h4>Model Results (n={len(model_df):,}, pseudo R-sq={logit_model.prsquared:.4f})</h4>"))
styled = results_df.style.format({
    'Odds Ratio': '{:.2f}', 'CI Low': '{:.2f}', 'CI High': '{:.2f}', 'p-value': '{:.4f}'
}).set_properties(**{'text-align': 'center'})
display(styled)

# Interpret results honestly
sig_any = results_df[results_df['p-value'] < 0.05]
marginal = results_df[(results_df['p-value'] >= 0.05) & (results_df['p-value'] < 0.10)]
nonsig = results_df[results_df['p-value'] >= 0.10]

display(HTML(
    "<p><strong>Model interpretation:</strong> The model explains very little variance "
    f"(pseudo R-squared = {logit_model.prsquared:.4f}), indicating that the factors we can "
    "extract from text are weak predictors of experience quality overall. "
    f"{'<strong>Positive Staff</strong> is the only predictor reaching traditional significance (p<0.05).' if len(sig_any) > 0 else ''}"
    f"{'<strong>Positive Staff</strong> is marginally significant (p<0.10), consistent with the significant pairwise test above.' if len(sig_any) == 0 and len(marginal[marginal['Predictor'] == 'Positive Staff']) > 0 else ''}"
    f"</p><p><strong>Not significant:</strong> {', '.join(nonsig['Predictor'].tolist())}</p>"
    f"<p><strong>Sensitivity:</strong> {sens_note}</p>"
    "<p><strong>Plain-language interpretation:</strong> No single factor we measured -- method, support "
    "system, fear level, or being alone -- significantly predicts whether someone will report a "
    "positive experience in this multivariate model. Positive staff interactions are the closest to "
    "significance, consistent with the pairwise finding above. The low pseudo R-squared tells us that "
    "the factors driving experience quality are either unmeasurable from text (e.g., gestational age, "
    "personal circumstances, pre-existing mental health) or too subtle for keyword-based extraction.</p>"
))


## Counterintuitive Findings Worth Investigating

We actively searched for results that contradict common assumptions about abortion experiences.

In [ ]:

findings = []

# 1. Fear does not predict outcomes
fear_yes = analyzable[analyzable['expresses_fear']]
fear_no = analyzable[~analyzable['expresses_fear']]
fear_pos_rate = fear_yes['positive_experience'].mean()
nofear_pos_rate = fear_no['positive_experience'].mean()
table_fear = [
    [int(fear_yes['positive_experience'].sum()), int((~fear_yes['positive_experience']).sum())],
    [int(fear_no['positive_experience'].sum()), int((~fear_no['positive_experience']).sum())]
]
or_fear, p_fear = fisher_exact(table_fear)
findings.append({
    'title': 'Fear and anxiety do not predict negative outcomes',
    'detail': (f"47% of analyzable users express fear or anxiety. Yet their positive experience rate "
               f"({fear_pos_rate:.1%}) is virtually identical to those who do not mention fear "
               f"({nofear_pos_rate:.1%}), Fisher's exact p={p_fear:.3f}. A clinician might expect "
               f"anxious patients to have worse experiences. Instead, fear appears to be a universal "
               f"aspect of the abortion experience that has no bearing on the outcome. This is "
               f"reassuring: being scared does not mean things will go badly.")
})

# 2. Method truly does not matter
findings.append({
    'title': 'Medical and surgical abortion produce essentially identical experience ratings',
    'detail': (f"Medical: {p1_ma:.1%} positive (n={len(ma_users)}) vs. Surgical: {p2_sa:.1%} "
               f"(n={len(sa_users)}), p={p_method:.3f}. Online forums are full of debates about "
               f"which method is 'better.' This data suggests the debate is misdirected. The "
               f"method itself is not what makes the experience good or bad.")
})

# 3. Support is not significant (surprising given common advice)
findings.append({
    'title': 'Having a support person does not significantly predict a positive experience',
    'detail': (f"Mentioning a support person: {p_sup:.1%} positive (n={len(sup_yes)}) vs. not "
               f"mentioning one: {p_nosup:.1%} (n={len(sup_no)}), p={p_support:.3f}. Common advice "
               f"emphasizes bringing someone with you. While intuitively sensible, this data does "
               f"not support the claim that having a support person measurably improves the "
               f"experience. Possible explanations: users who mention partners or family may be "
               f"discussing relationship stress rather than support; the benefit of support may be "
               f"real but too small to detect at this sample size; or support quality matters more "
               f"than support presence.")
})

html_f = ""
for i, f in enumerate(findings, 1):
    html_f += (f"<div style='background:#fff3cd; padding:12px; border-left:4px solid #ffc107; "
               f"margin:10px 0; border-radius:4px;'>"
               f"<strong>{i}. {f['title']}</strong><br>{f['detail']}</div>")
display(HTML(html_f))


## What Patients Are Saying *(experimental -- under development)*

The following quotes are drawn from posts in the dataset. They illustrate the quantitative findings above. Dates are included for temporal context.

In [ ]:

def clean_quote(text, max_words=40):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\*+', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = text.strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    for s in sentences:
        words = s.split()
        if 10 <= len(words) <= max_words:
            return s.strip()
    if sentences and len(sentences[0].split()) >= 5:
        return ' '.join(sentences[0].split()[:max_words])
    return None

quote_posts = pd.read_sql(
    "SELECT p.user_id, p.body_text, date(p.post_date, 'unixepoch') as post_date "
    "FROM posts p WHERE LENGTH(p.body_text) > 100 "
    "AND p.body_text NOT LIKE '%Welcome to /r/abortion%' "
    "AND p.body_text NOT LIKE '%Abortion Resolution Workbook%' "
    "ORDER BY p.post_date DESC", conn
)
bt_q = quote_posts['body_text'].str.lower()

categories_q = [
    ("Staff quality predicts positive outcomes",
     quote_posts[bt_q.str.contains('kind|gentle|compassionate|caring|nice nurse|wonderful staff|amazing', regex=True, na=False)
                 & bt_q.str.contains('relief|relieved|positive|grateful|glad', regex=True, na=False)]),
    ("Fear does not preclude a positive experience",
     quote_posts[bt_q.str.contains('scared|terrified|afraid|nervous', regex=True, na=False)
                 & bt_q.str.contains('relief|relieved|not as bad|not that bad|turned out', regex=True, na=False)]),
    ("Being alone is difficult but survivable",
     quote_posts[bt_q.str.contains('by myself|alone|secret|no one knew', regex=True, na=False)
                 & bt_q.str.contains('relief|okay|fine|manageable|got through|made it', regex=True, na=False)]),
    ("Support does not guarantee a positive experience (complicating the narrative)",
     quote_posts[bt_q.str.contains('partner|boyfriend|husband|mom|family|support', regex=True, na=False)
                 & bt_q.str.contains('regret|guilt|sad|cry|crying|difficult|hard|struggle|traumat', regex=True, na=False)]),
    ("Physical pain shapes perception",
     quote_posts[bt_q.str.contains('worst pain|excruciating|screaming|agony|unbearable|so much pain', regex=True, na=False)]),
]

html_quotes = ""
for cat_name, df_q in categories_q:
    if len(df_q) > 0:
        sample = df_q.sample(min(1, len(df_q)), random_state=42).iloc[0]
        quote = clean_quote(sample['body_text'])
        if quote and len(quote) > 20:
            html_quotes += (
                f"<div style='border-left:3px solid #3498db; padding:8px 12px; margin:8px 0; "
                f"background:#f8f9fa;'><strong>{cat_name}</strong><br>"
                f'<em>"{quote}"</em><br>'
                f"<small style='color:#666;'>-- r/abortion user, {sample['post_date']}</small></div>"
            )

if html_quotes:
    display(HTML(html_quotes))
else:
    display(HTML("<p>Could not find quotes meeting quality criteria for all categories.</p>"))


## Recommendations

Based on pairwise tests and the logistic regression, we categorize findings into evidence tiers. Given the largely null results, the recommendation structure reflects what *did not* matter as much as what did.

In [ ]:

# Tiered recommendations
strong_recs = []
moderate_recs = []
preliminary_recs = []
null_findings = []

# Staff positive: significant in pairwise (p=0.029), marginal in regression (p~0.06)
staff_pos_n = int(model_df['staff_pos'].sum())
staff_pos_or_pairwise = or_staff
staff_pos_p_pairwise = p_staff
staff_pos_or_reg = results_df[results_df['Predictor'] == 'Positive Staff']['Odds Ratio'].values[0]
staff_pos_p_reg = results_df[results_df['Predictor'] == 'Positive Staff']['p-value'].values[0]

if staff_pos_p_pairwise < 0.05 and staff_pos_n >= 30:
    strong_recs.append(
        f"Positive staff interactions increase odds of positive experience "
        f"(pairwise OR={staff_pos_or_pairwise:.2f}, p={staff_pos_p_pairwise:.3f}, n={staff_pos_n}; "
        f"regression OR={staff_pos_or_reg:.2f}, p={staff_pos_p_reg:.3f})"
    )

# Everything else: not significant
null_findings.append(f"Abortion method (medical vs surgical): p={p_method:.3f}, no difference")
null_findings.append(f"Having a support person: p={p_support:.3f}, no significant effect")
null_findings.append(f"Expressing fear/anxiety: p={p_fear:.3f}, no predictive value")

fig, ax = plt.subplots(figsize=(10, 4))
all_items = []
all_colors_r = []

for r in strong_recs:
    all_items.append(('Strong', r.split(' (')[0]))
    all_colors_r.append('#27ae60')
for r in moderate_recs:
    all_items.append(('Moderate', r.split(' (')[0]))
    all_colors_r.append('#f39c12')
for r in null_findings:
    all_items.append(('Null', r.split(':')[0] if ':' in r else r))
    all_colors_r.append('#95a5a6')

if all_items:
    y_pos = range(len(all_items))
    ax.barh(list(y_pos), [1]*len(all_items), color=all_colors_r, height=0.6)
    for yp, (tier, label) in zip(y_pos, all_items):
        ax.text(0.02, yp, f"[{tier}] {label}", va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 1.2)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_title('Evidence Summary')
    for spine in ax.spines.values():
        spine.set_visible(False)
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#27ae60', label='Strong (n>=30, p<0.05)'),
        Patch(facecolor='#95a5a6', label='Null finding (p>0.05)')
    ]
    ax.legend(handles=legend_elements, loc='lower right', frameon=True)
    plt.tight_layout()
    plt.show()

display(HTML(
    "<h4>Strong Evidence (n>=30, p<0.05 in pairwise test)</h4>"
    f"<ul>{''.join(f'<li>{r}</li>' for r in strong_recs) if strong_recs else '<li>None</li>'}</ul>"
    "<h4>Null Findings (not statistically significant)</h4>"
    f"<ul>{''.join(f'<li>{r}</li>' for r in null_findings)}</ul>"
))


## Conclusion

In [ ]:

overall_pos_rate = analyzable['positive_experience'].mean()

display(HTML(
    "<div style='line-height:1.8;'>"
    "<p>This analysis set out to answer a question many patients ask: does it matter whether I "
    "choose medical or surgical abortion? Should I focus on finding the right clinic, or on "
    "having someone with me?</p>"

    f"<p>The answer is more nuanced than expected. Based on {len(analyzable):,} users with "
    f"analyzable experience signals from r/abortion, <strong>most of the factors patients worry "
    f"about do not significantly predict experience quality</strong>. Method choice (medical vs. "
    f"surgical) showed virtually identical positive experience rates. Having a support person "
    f"trended positive but did not reach significance. Even fear and anxiety -- expressed by "
    f"nearly half the community -- had no bearing on outcomes.</p>"

    "<p>The one exception is <strong>staff quality</strong>. Users who described kind, compassionate "
    "staff were significantly more likely to report a positive experience (45.6% vs. 34.2%, p=0.029). "
    "This was the only factor to reach statistical significance in pairwise testing, and the strongest "
    "predictor in the multivariate model (though marginal at p=0.06 after controlling for other factors). "
    "The effect is modest -- roughly 1 additional positive experience per 9 patients who encounter "
    "good staff -- but it is the only modifiable factor that showed a measurable association.</p>"

    "<p>For a patient facing an abortion decision, this data offers a counterintuitive message: "
    "<strong>stop agonizing over the method</strong>. Medical and surgical produce equivalent experience "
    "ratings. Instead, prioritize clinic selection -- specifically, clinics known for compassionate "
    "patient interaction. The data cannot tell us whether good staff cause better experiences "
    "or whether patients who have good experiences are more likely to remember staff positively. "
    "But the association is there, and it is the only one that survived statistical testing.</p>"

    "<p>The most honest takeaway may be the humbling one: at this sample size, with text-based "
    "measures, <strong>we cannot reliably predict who will have a good experience</strong>. The "
    "factors that matter most -- personal resilience, life circumstances, gestational age, access "
    "barriers -- may not be extractable from Reddit posts. What we can say is that the things "
    "patients are told to worry about (method, support person, fear) do not appear to determine "
    "outcomes in this data.</p>"
    "</div>"
))


## Research Limitations

This analysis has several important limitations that contextualize all findings.

In [ ]:

display(HTML(
    "<div style='line-height:1.7;'><ol>"
    "<li><strong>Selection bias:</strong> r/abortion users are not representative of all people who "
    "have abortions. The community skews toward those who seek online support, are English-speaking, "
    "and have internet access. People who had uncomplicated experiences may never post.</li>"
    "<li><strong>Reporting bias:</strong> Users choose what to share. Posts emphasizing emotional "
    "distress or strong opinions are likely overrepresented compared to neutral experiences. The "
    "community may also have norms that encourage certain narratives.</li>"
    "<li><strong>Survivorship bias:</strong> We only observe users who continued to use Reddit after "
    "their experience. Those who found the experience so traumatic they withdrew from online "
    "communities are absent from this data.</li>"
    "<li><strong>Recall bias:</strong> Posts written weeks or months after the procedure may reflect "
    "reconstructed memories rather than real-time experience. Emotional states at the time of "
    "posting may color recollections.</li>"
    "<li><strong>Confounding:</strong> The relationship between staff quality and positive outcomes "
    "could be confounded by clinic quality (better clinics have better staff AND better medical "
    "outcomes), insurance status, and geographic access. We cannot isolate the causal effect of "
    "staff behavior from these correlated factors.</li>"
    "<li><strong>No control group:</strong> There is no comparison to people who considered but did "
    "not have an abortion, or to the general population's experience with other medical procedures. "
    "'Positive experience' is defined relative to this community's posts, not to an external "
    "benchmark.</li>"
    "<li><strong>Sentiment vs. efficacy:</strong> Our text-based outcome measures (relief, regret, "
    "pain) are sentiment indicators, not clinical outcomes. A user who writes 'relief' may be "
    "expressing a momentary feeling rather than a sustained emotional state. We cannot measure "
    "clinical complications, long-term mental health, or reproductive outcomes.</li>"
    "<li><strong>Temporal snapshot:</strong> This data covers one month (March 13 -- April 12, 2026). "
    "Seasonal patterns, news events, and policy changes (particularly relevant for abortion access "
    "in the US) may influence community composition and sentiment. Findings may not generalize to "
    "other time periods.</li>"
    "</ol></div>"
))


In [ ]:

display(HTML(
    '<p style="font-size: 1.2em; font-weight: bold; font-style: italic;">'
    '"These findings reflect reporting patterns in online communities, not population-level '
    'treatment effects. This is not medical advice."</p>'
))
